In [24]:
import pandas as pd
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

# 1. Load all files once
dim_stores = pd.read_csv("dim_stores.csv")
dim_skus = pd.read_csv("dim_skus.csv")
dim_suppliers = pd.read_csv("dim_suppliers.csv",
                            na_values=["N/A","missing","--","NA","null"],
                            keep_default_na=True)
dim_events = pd.read_csv("dim_events.csv")
fact = pd.read_csv("fact_inventory_daily.csv")

# Merge supplier reliability score into fact table
fact = fact.merge(dim_suppliers[['supplier_id', 'reliability_score']], on='supplier_id', how='left')
# Merge perishable information from dim_skus into fact table
fact = fact.merge(dim_skus[['sku_id', 'is_perishable']], on='sku_id', how='left')
# Merge category information from dim_skus into fact table
fact = fact.merge(dim_skus[['sku_id', 'category']], on='sku_id', how='left')


In [25]:
from sklearn.model_selection import train_test_split
y = fact['stockout_risk']

X = fact.drop(['stockout_risk', 'date'], axis=1)

categorical_cols = X.select_dtypes(include='object').columns
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Random Forest with class weights
clf = RandomForestClassifier(class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))




              precision    recall  f1-score   support

     At-Risk       0.88      0.94      0.90      1037
    Imminent       0.86      0.74      0.79       457
        Safe       1.00      0.99      0.99      2826

    accuracy                           0.95      4320
   macro avg       0.91      0.89      0.90      4320
weighted avg       0.95      0.95      0.95      4320



In [26]:
#  Row counts sanity check
print("dim_stores.csv:", dim_stores.shape[0], "rows")
print("dim_skus.csv:", dim_skus.shape[0], "rows")
print("dim_suppliers.csv:", dim_suppliers.shape[0], "rows")
print("dim_events.csv:", dim_events.shape[0], "rows")
print("fact_inventory_daily.csv:", fact.shape[0], "rows (12 x 60 x 30)")

# Target distribution sanity check
target_counts = fact["stockout_risk"].value_counts()
target_pct = fact["stockout_risk"].value_counts(normalize=True) * 100
print("\nTarget distribution (fact_inventory_daily.csv):")
for risk in ["Safe","At-Risk","Imminent"]:
    print(f" {risk}: {target_pct[risk]:.2f}% ({target_counts[risk]} rows)")

#  Date range sanity check
print("\nDate range:", fact["date"].min(), "to", fact["date"].max(),
      f"({fact['date'].nunique()} unique dates)")
print("Diwali Week: Oct 22-26, 2026 (demand multiplier peaks at 2.6x on Oct 25)")
print("Weekend Flash Sale: Oct 11, 2026 (1.3x demand, all categories)")

# Festival week effect
diwali_days = ["2026-10-22","2026-10-23","2026-10-24","2026-10-25","2026-10-26"]
festival = fact[fact["date"].isin(diwali_days)]
non_festival = fact[~fact["date"].isin(diwali_days)]
imminent_rate_non_festival = (non_festival["stockout_risk"]=="Imminent").mean()*100
imminent_rate_festival = (festival["stockout_risk"]=="Imminent").mean()*100
print("\nFestival week effect:")
print(" Imminent rate, non-festival days: %.2f%%" % imminent_rate_non_festival)
print(" Imminent rate, festival week: %.2f%% (%.2fx spike)" %
      (imminent_rate_festival, imminent_rate_festival/imminent_rate_non_festival))

# Supplier reliability effect
fact["reliability_score"] = pd.to_numeric(fact["reliability_score"], errors="coerce")
low = fact[fact["reliability_score"] < 0.75]
mid = fact[(fact["reliability_score"] >= 0.75) & (fact["reliability_score"] < 0.85)]
high = fact[fact["reliability_score"] >= 0.85]
print("\nSupplier reliability effect:")
print(" Imminent rate, low reliability (<0.75): %.2f%%" %
      ((low["stockout_risk"]=="Imminent").mean()*100))
print(" Imminent rate, mid reliability (0.75-0.85): %.2f%%" %
      ((mid["stockout_risk"]=="Imminent").mean()*100))
print(" Imminent rate, high reliability (>=0.85): %.2f%%" %
      ((high["stockout_risk"]=="Imminent").mean()*100))

# Perishable effect
perishable = fact[fact["is_perishable"]=="Y"]
non_perishable = fact[fact["is_perishable"]=="N"]
print("\nPerishable effect (smaller but real):")
print(" Imminent rate, non-perishable SKUs: %.1f%%" %
      ((non_perishable["stockout_risk"]=="Imminent").mean()*100))
print(" Imminent rate, perishable SKUs: %.1f%%" %
      ((perishable["stockout_risk"]=="Imminent").mean()*100))

#  Missing lead_time_days_actual
missing_lead_time = fact["lead_time_days_actual"].isna().sum()
print("\nMissing lead_time_days_actual: %d of %d rows (%.1f%%)" %
      (missing_lead_time, fact.shape[0], missing_lead_time/fact.shape[0]*100))
print(" — by construction, only populated on days a reorder was placed")

dim_stores.csv: 12 rows
dim_skus.csv: 60 rows
dim_suppliers.csv: 15 rows
dim_events.csv: 30 rows
fact_inventory_daily.csv: 21600 rows (12 x 60 x 30)

Target distribution (fact_inventory_daily.csv):
 Safe: 65.42% (14131 rows)
 At-Risk: 24.01% (5186 rows)
 Imminent: 10.57% (2283 rows)

Date range: 2026-10-01 to 2026-10-30 (30 unique dates)
Diwali Week: Oct 22-26, 2026 (demand multiplier peaks at 2.6x on Oct 25)
Weekend Flash Sale: Oct 11, 2026 (1.3x demand, all categories)

Festival week effect:
 Imminent rate, non-festival days: 9.30%
 Imminent rate, festival week: 16.92% (1.82x spike)

Supplier reliability effect:
 Imminent rate, low reliability (<0.75): 15.83%
 Imminent rate, mid reliability (0.75-0.85): 3.26%
 Imminent rate, high reliability (>=0.85): 3.82%

Perishable effect (smaller but real):
 Imminent rate, non-perishable SKUs: 9.3%
 Imminent rate, perishable SKUs: 12.8%

Missing lead_time_days_actual: 19899 of 21600 rows (92.1%)
 — by construction, only populated on days a reord

In [28]:

def clean_cols(df):
    df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]
    return df

dim_stores = clean_cols(dim_stores)
dim_skus = clean_cols(dim_skus)
dim_suppliers = clean_cols(dim_suppliers)
dim_events = clean_cols(dim_events)
fact = clean_cols(fact)

if 'supplier_id' in dim_skus.columns:
    dim_skus.rename(columns={'supplier_id': 'sku_primary_supplier_id'}, inplace=True)

# --- 2. Clean dimension tables ---
dim_stores["city_display"] = dim_stores["city_display"].str.title()

# Handle supplier reliability
dim_suppliers["reliability_score"] = pd.to_numeric(dim_suppliers["reliability_score"], errors="coerce")
dim_suppliers["reliability_score"] = dim_suppliers.groupby("categories_supplied")["reliability_score"]\
    .transform(lambda x: x.fillna(x.median()))

# Convert date columns to datetime objects for consistent merging
dim_events['date'] = pd.to_datetime(dim_events['date'])
fact['date'] = pd.to_datetime(fact['date']) # Convert fact['date'] to datetime

# --- 3. Join tables step by step ---
fact_full = fact.merge(dim_stores, on="store_id", how="left") \
                .merge(dim_skus, on="sku_id", how="left") \
                .merge(dim_suppliers, on="supplier_id", how="left") \
                .merge(dim_events, on="date", how="left")

# --- 4. Verify row counts ---
print("Final dataset rows:", fact_full.shape[0])   # should be 21,600
print("Expected rows: 21,600")

# --- 5. Quick sanity check ---
print(fact_full.head())

Final dataset rows: 21600
Expected rows: 21,600
        date store_id  sku_id supplier_id  opening_stock  units_demanded  \
0 2026-10-01     ST01  SKU001       SUP11          159.9              15   
1 2026-10-02     ST01  SKU001       SUP11          144.9              10   
2 2026-10-03     ST01  SKU001       SUP11          134.9              15   
3 2026-10-04     ST01  SKU001       SUP11          119.9              10   
4 2026-10-05     ST01  SKU001       SUP11          109.9               9   

   units_sold  closing_stock  reorder_point reorder_placed  ...  \
0        15.0          144.9           36.4              N  ...   
1        10.0          134.9           36.4              N  ...   
2        15.0          119.9           36.4              N  ...   
3        10.0          109.9           36.4              N  ...   
4         9.0          100.9           36.4              N  ...   

   sku_primary_supplier_id          supplier_name  \
0                    SUP11  Coastal Foo

In [29]:
# Feature Engineering
fact["reorder_gap"] = fact["reorder_point"] - fact["closing_stock"]
fact["days_of_cover_ratio"] = fact["days_of_cover"] / fact["lead_time_days_expected"]
fact["is_recent_reorder"] = fact.groupby(["store_id","sku_id"])["reorder_placed"].shift(1).fillna("N")
fact = pd.get_dummies(fact, columns=["category"], prefix="cat")
fact["date"] = pd.to_datetime(fact["date"])
fact["day_of_month"] = fact["date"].dt.day
festival_start = pd.to_datetime("2026-10-22")
fact["days_since_festival_start"] = (fact["date"] - festival_start).dt.days.clip(lower=0)

# Train/Test Split (time-based)
train = fact[fact["date"] <= "2026-10-23"]
test = fact[fact["date"] >= "2026-10-24"]
feature_cols = ["opening_stock","closing_stock","units_demanded","units_sold",
                "sales_velocity_7d","days_of_cover","reorder_gap","days_of_cover_ratio",
                "reliability_score","day_of_month","days_since_festival_start"] + \
                [c for c in fact.columns if c.startswith("cat_")]
X_train, y_train = train[feature_cols].copy(), train["stockout_risk"].copy()
X_test, y_test = test[feature_cols].copy(), test["stockout_risk"].copy()

# Impute missing values in X_train and X_test using the median
for col in X_train.columns:
    if X_train[col].dtype != 'object': # Impute only numerical columns
        median_val = X_train[col].median()
        X_train.loc[:, col] = X_train[col].fillna(median_val)
        X_test.loc[:, col] = X_test[col].fillna(median_val)

# Scale numerical features
scaler = StandardScaler()
numerical_cols = X_train.select_dtypes(include=np.number).columns
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

#  Baseline model
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
print("\nBaseline (majority class):")
print(classification_report(y_test, baseline.predict(X_test)))

#  Logistic Regression
logreg = LogisticRegression(max_iter=1000, multi_class="multinomial")
logreg.fit(X_train, y_train)
print("\nMultinomial Logistic Regression:")
print(classification_report(y_test, logreg.predict(X_test)))

#  Random Forest
rf = RandomForestClassifier(class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
print("\nRandom Forest:")
print(classification_report(y_test, rf.predict(X_test)))

#  Gradient Boosting
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
print("\nGradient Boosting:")
print(classification_report(y_test, gb.predict(X_test)))

#  Confusion Matrix (focus on Imminent recall)
cm = confusion_matrix(y_test, rf.predict(X_test), labels=["Safe","At-Risk","Imminent"])
print("\nConfusion Matrix (Random Forest):")
print(cm)


Baseline (majority class):
              precision    recall  f1-score   support

     At-Risk       0.00      0.00      0.00      1126
    Imminent       0.00      0.00      0.00       775
        Safe       0.62      1.00      0.77      3139

    accuracy                           0.62      5040
   macro avg       0.21      0.33      0.26      5040
weighted avg       0.39      0.62      0.48      5040



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/l


Multinomial Logistic Regression:
              precision    recall  f1-score   support

     At-Risk       0.93      0.56      0.70      1126
    Imminent       0.65      0.96      0.77       775
        Safe       0.97      0.99      0.98      3139

    accuracy                           0.89      5040
   macro avg       0.85      0.84      0.82      5040
weighted avg       0.91      0.89      0.89      5040


Random Forest:
              precision    recall  f1-score   support

     At-Risk       0.78      0.95      0.86      1126
    Imminent       0.90      0.62      0.74       775
        Safe       1.00      1.00      1.00      3139

    accuracy                           0.93      5040
   macro avg       0.90      0.86      0.86      5040
weighted avg       0.94      0.93      0.93      5040


Gradient Boosting:
              precision    recall  f1-score   support

     At-Risk       0.85      0.94      0.89      1126
    Imminent       0.90      0.76      0.82       775
     